# Customer Churn & Retention Analytics with AI
## IBM Telco Customer Churn Dataset — End-to-End Machine Learning Project

**Author:** Kadeejath Muhsina  
**Dataset:** IBM Telco Customer Churn (7,043 customers, 21 features)  
**Python Version:** 3.13.7  

---

### Project Overview

This notebook implements a complete, production-quality customer churn analytics pipeline that:
- Loads and cleans the IBM Telco Customer Churn dataset
- Performs exploratory data analysis with 5 chart categories
- Engineers 6 domain-specific features
- Trains and compares Logistic Regression, Random Forest, and XGBoost classifiers
- Optimises the best model with RandomizedSearchCV (40 iterations)
- Finds the optimal classification threshold using F1-score sweep
- Predicts churn probability and segments customers into Low / Medium / High risk
- Explains model decisions with SHAP (SHapley Additive exPlanations)
- Generates personalised retention recommendations with a priority score
- Exports a Power BI-ready Excel workbook (8 formatted sheets)

---

### Pipeline Steps

| Step | Description |
|------|-------------|
| 0 | Environment setup & dataset loading |
| 1 | Data understanding & cleaning |
| 2 | Exploratory data analysis |
| 3 | Feature engineering & preprocessing |
| 4 | Model training — LR, RF, XGBoost |
| 5 | Model comparison & cross-validation |
| 6 | Hyperparameter optimisation |
| 7 | Threshold analysis |
| 8 | Churn probability prediction & risk segmentation |
| 9 | SHAP explainability |
| 10 | Retention recommendations |
| 11 | Power BI data export |
| 12 | Results summary |


---
## Step 0 · Environment Setup

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings('ignore')

# Add project root to path so src/ modules can be imported
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import xgboost as xgb
import shap
import joblib

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

print('Library versions:')
print(f'  Python      : {sys.version.split()[0]}')
print(f'  pandas      : {pd.__version__}')
print(f'  numpy       : {np.__version__}')
print(f'  matplotlib  : {matplotlib.__version__}')
print(f'  scikit-learn: {sklearn.__version__}')
print(f'  xgboost     : {xgb.__version__}')
print(f'  shap        : {shap.__version__}')
print('\nEnvironment ready.')

In [ ]:
# Download dataset (tries GitHub mirror; falls back to synthetic if offline)
import subprocess
result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_ROOT, 'data', 'download_dataset.py')],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

---
## Step 1 · Data Understanding & Cleaning

The raw IBM Telco dataset has **7,043 rows and 21 columns**. Key issues addressed:
- `TotalCharges` is stored as object dtype (whitespace for 11 new customers with tenure=0)
- Target column `Churn` is text ('Yes'/'No') — encoded to int (1/0)
- `SeniorCitizen` is already binary (0/1)
- No true missing values after TotalCharges correction

In [ ]:
from src.config import RAW_DATA_PATH, TARGET_COL

df_raw = pd.read_csv(RAW_DATA_PATH)
print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(5)

In [ ]:
# Data types
print('=== Data Types ===')
print(df_raw.dtypes)
print()

# Missing values — TotalCharges has whitespace entries
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce')
print(f'=== Missing after coerce ===')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print()

# Class balance
print('=== Target Distribution ===')
vc = df_raw[TARGET_COL].value_counts()
print(vc)
print(f'Churn rate: {vc["Yes"]/vc.sum()*100:.1f}%')

In [ ]:
# Descriptive statistics for numeric columns
df_raw[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']].describe().round(2)

In [ ]:
# Run full automated cleaning pipeline
# Fixes TotalCharges (11 NaN -> median), encodes target, removes duplicates
from src.data_cleaning import clean
df = clean()
print(f'\nCleaned shape: {df.shape}')
print(f'Target distribution (0=No Churn, 1=Churn):')
print(df[TARGET_COL].value_counts())
df.head(3)

---
## Step 2 · Exploratory Data Analysis

**Key findings from EDA:**
- Overall churn rate: **26.5%** (1,869 of 7,043 customers)
- Month-to-month contracts churn at **42.7%** vs 2.8% for two-year contracts
- Fiber optic internet users churn at **41.9%** — highest among internet service types
- Electronic check payment method has the highest churn rate: **45.3%**
- Customers in the first 12 months (tenure 0–12m) churn at **47.4%**
- Churn drops dramatically after 24 months of tenure

In [ ]:
# --- 2.1 Churn distribution ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = df[TARGET_COL].value_counts().sort_index()
labels = ['No Churn (0)', 'Churn (1)']
colors = ['#3b82d4', '#e05c5c']

axes[0].bar(labels, counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Churn Count', fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for bar, v in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+40, f'{v:,}', ha='center', fontsize=10)

axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Churn Split (%)', fontweight='bold')

plt.suptitle('Customer Churn Distribution — IBM Telco (n=7,043)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.2 Numeric feature distributions by churn status ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

for ax, col in zip(axes, numeric_cols):
    for val, label, color in [(0, 'Retained', '#3b82d4'), (1, 'Churned', '#e05c5c')]:
        ax.hist(df[df[TARGET_COL]==val][col], bins=30, alpha=0.6,
                color=color, label=label, edgecolor='none')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Numeric Feature Distributions by Churn Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.3 Churn rate by key categorical features ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(
    axes,
    ['Contract', 'InternetService', 'PaymentMethod'],
    ['Churn Rate by Contract Type', 'Churn Rate by Internet Service', 'Churn Rate by Payment Method']
):
    rates = df.groupby(col)[TARGET_COL].mean().sort_values() * 100
    colors_bar = ['#e05c5c' if v == rates.max() else '#3b82d4' for v in rates.values]
    bars = ax.barh(rates.index.astype(str), rates.values,
                   color=colors_bar, edgecolor='white', alpha=0.9)
    for bar, v in zip(bars, rates.values):
        ax.text(v+0.5, bar.get_y()+bar.get_height()/2,
                f'{v:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('Churn Rate (%)')
    ax.set_title(title, fontweight='bold')
    ax.axvline(df[TARGET_COL].mean()*100, color='gray', linestyle='--',
               linewidth=1, label=f'Avg {df[TARGET_COL].mean()*100:.1f}%')
    ax.legend(fontsize=8)

plt.suptitle('Churn Rate by Service Characteristics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.4 Churn rate by tenure cohort ---
df2 = df.copy()
bins   = [0, 12, 24, 36, 48, 60, 72]
labels = ['0-12m', '13-24m', '25-36m', '37-48m', '49-60m', '61-72m']
df2['Tenure Cohort'] = pd.cut(df2['tenure'], bins=bins, labels=labels, include_lowest=True)
cohort = df2.groupby('Tenure Cohort', observed=True)[TARGET_COL].mean() * 100

fig, ax = plt.subplots(figsize=(9, 4))
colors_c = ['#e05c5c' if v > 30 else '#f59e0b' if v > 20 else '#3b82d4' for v in cohort.values]
bars = ax.bar(cohort.index.astype(str), cohort.values, color=colors_c, edgecolor='white', alpha=0.9)
for bar, v in zip(bars, cohort.values):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.1f}%',
            ha='center', va='bottom', fontsize=9)
ax.axhline(df[TARGET_COL].mean()*100, color='#7c5cd8', linestyle='--',
           linewidth=1.5, label=f'Overall Avg: {df[TARGET_COL].mean()*100:.1f}%')
ax.set_xlabel('Tenure Cohort', fontsize=11)
ax.set_ylabel('Churn Rate (%)', fontsize=11)
ax.set_title('Churn Rate by Tenure Cohort', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nTenure Cohort Churn Rates:')
for band, rate in cohort.items():
    print(f'  {band}: {rate:.1f}%')

In [ ]:
# --- 2.5 Correlation heatmap (numeric features) ---
numeric_df = df.select_dtypes(include='number')
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(numeric_df.corr(), dtype=bool))
sns.heatmap(numeric_df.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', vmax=1, vmin=-1, center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink':0.8}, ax=ax)
ax.set_title('Numeric Feature Correlation Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3 · Feature Engineering & Preprocessing

**Encoding:**
- Binary columns (Yes/No, Male/Female) mapped to 1/0
- Multi-category columns (Contract, InternetService, PaymentMethod, etc.) one-hot encoded with `drop_first=True`

**Derived Features (6 new features):**

| Feature | Formula | Rationale |
|---------|---------|----------|
| `avg_monthly_spend` | TotalCharges / tenure | Actual average spend rate |
| `charge_per_month_ratio` | MonthlyCharges / (tenure+1) | Spend relative to tenure |
| `num_services` | Count of active add-ons | Service stickiness indicator |
| `tenure_band` | Ordinal bucket (0–5) | Non-linear tenure effect |
| `is_high_value` | 1 if charges ≥ 75th pct | High-value customer flag |
| `is_long_term` | 1 if tenure ≥ 24m | Loyalty signal |

**Train/Test Split:** 80% train (5,634) / 20% test (1,409) — stratified by churn label  
**Scaling:** `StandardScaler` fitted on training data only (no data leakage)

In [ ]:
from src.feature_engineering import run_feature_engineering

X_train, X_test, y_train, y_test, X_tr_raw, X_te_raw = run_feature_engineering()

print(f'Training set : {X_train.shape[0]:,} rows x {X_train.shape[1]} features')
print(f'Test set     : {X_test.shape[0]:,} rows x {X_test.shape[1]} features')
print(f'Churn rate - train: {y_train.mean():.2%}  |  test: {y_test.mean():.2%}')
print(f'\nFeature columns ({X_train.shape[1]} total):')
print(list(X_train.columns))

In [ ]:
# Top 20 features by absolute Pearson correlation with churn
corrs = X_train.corrwith(y_train).abs().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 7))
colors_feat = ['#e05c5c' if v > 0.15 else '#3b82d4' for v in corrs.values[::-1]]
ax.barh(corrs.index[::-1], corrs.values[::-1], color=colors_feat, edgecolor='none')
ax.axvline(0.15, color='#7c5cd8', linestyle='--', linewidth=1.2, label='|r| = 0.15 threshold')
ax.set_xlabel('|Pearson r| with Churn', fontsize=11)
ax.set_title('Top 20 Features — Correlation with Churn', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('Top 10:')
for feat, val in corrs.head(10).items():
    print(f'  {feat:<45s}  r = {val:.4f}')

---
## Step 4 · Model Training

Three classifiers trained with class-imbalance handling:

| Model | Imbalance Strategy | Key Hyperparameters |
|-------|-------------------|--------------------|
| Logistic Regression | `class_weight='balanced'` | C=1.0, L2, lbfgs solver |
| Random Forest | `class_weight='balanced'` | 300 trees, max_depth=8 |
| XGBoost | `scale_pos_weight=2.77` | 300 trees, lr=0.05, depth=5 |

Evaluation uses 5-fold stratified cross-validation plus a held-out 20% test set.

In [ ]:
from src.model_training import train_models

results_df, cv_df, trained_models = train_models(X_train, X_test, y_train, y_test)

print('\n=== Test-Set Performance ===' )
display(results_df[['Model','ROC-AUC','PR-AUC','F1','Precision','Recall','Accuracy']])

In [ ]:
print('=== 5-Fold Cross-Validation Results ===')
display(cv_df)

---
## Step 5 · Model Comparison

**Actual test-set results (1,409 held-out samples):**

| Model | ROC-AUC | PR-AUC | F1 | Precision | Recall | Accuracy |
|-------|---------|--------|----|-----------|--------|----------|
| **Logistic Regression** | **0.8461** | **0.6664** | 0.6138 | 0.5079 | 0.7754 | 0.7410 |
| Random Forest | 0.8434 | 0.6552 | **0.6265** | 0.5184 | **0.7914** | 0.7495 |
| XGBoost | 0.8363 | 0.6418 | 0.6198 | **0.5261** | 0.7540 | **0.7544** |

**Selected model: Logistic Regression** — highest ROC-AUC (0.8461) and PR-AUC (0.6664).  
For churn prediction, high Recall is critical — we prefer to err on the side of flagging customers for retention outreach.

In [ ]:
# Visual comparison of all models across all metrics
metrics = ['ROC-AUC', 'PR-AUC', 'F1', 'Precision', 'Recall', 'Accuracy']
x = np.arange(len(metrics))
width = 0.25
model_colors = {'Logistic Regression': '#3b82d4', 'Random Forest': '#7c5cd8', 'XGBoost': '#e05c5c'}

fig, ax = plt.subplots(figsize=(13, 5))
for i, (_, row) in enumerate(results_df.iterrows()):
    name = row['Model']
    vals = [row[m] for m in metrics]
    offset = (i - 1) * width
    bars = ax.bar(x + offset, vals, width, label=name,
                  color=model_colors.get(name, '#999'), edgecolor='white', alpha=0.9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison — Test Set (n=1,409)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Display saved ROC and PR curve chart
from IPython.display import Image, display as ipy_display
ipy_display(Image(os.path.join(PROJECT_ROOT, 'reports', '07_roc_pr_curves.png')))

In [ ]:
# Display saved confusion matrices
ipy_display(Image(os.path.join(PROJECT_ROOT, 'reports', '08_confusion_matrices.png')))

---
## Step 6 · Hyperparameter Optimisation

**Method:** `RandomizedSearchCV` with 5-fold stratified CV, scoring = ROC-AUC  
**Search space (Logistic Regression):** C ∈ {0.001, 0.01, 0.1, 1, 10, 100}, penalty ∈ {L1, L2}, solver ∈ {liblinear, saga}  
**Best parameters found:** `C=0.1, penalty='l1', solver='liblinear'`  
**Best CV ROC-AUC:** 0.8485

In [ ]:
from src.model_optimization import run_optimization

# n_iter=10 for notebook speed; full pipeline uses n_iter=40
optimised_model, optimal_threshold = run_optimization(
    results_df, trained_models, X_train, X_test, y_train, y_test, n_iter=10
)
print(f'\nOptimal classification threshold: {optimal_threshold:.2f}')

---
## Step 7 · Threshold Analysis

Default threshold of 0.50 is not optimal for imbalanced churn prediction.  
We sweep thresholds 0.05–0.95 and choose the one maximising F1.

**Result: Optimal threshold = 0.57**  
At t=0.57: Precision=0.544, Recall=0.741, F1=0.627, Accuracy=0.767

In [ ]:
# Load threshold analysis results
thresh_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'reports', 'threshold_analysis.csv'))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df['Threshold'], thresh_df['Precision'], color='#3b82d4', lw=2, label='Precision')
ax.plot(thresh_df['Threshold'], thresh_df['Recall'],    color='#e05c5c', lw=2, label='Recall')
ax.plot(thresh_df['Threshold'], thresh_df['F1'],        color='#7c5cd8', lw=2.5, label='F1 Score')
ax.axvline(optimal_threshold, color='gray', linestyle='--', lw=1.5,
           label=f'Optimal t={optimal_threshold:.2f}')
ax.set_xlabel('Classification Threshold', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Threshold Analysis — Precision / Recall / F1', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Show optimal row
opt_row = thresh_df.loc[thresh_df['F1'].idxmax()]
print(f'Best threshold: t={opt_row["Threshold"]:.2f}')
print(f'  Precision = {opt_row["Precision"]:.4f}')
print(f'  Recall    = {opt_row["Recall"]:.4f}')
print(f'  F1        = {opt_row["F1"]:.4f}')
print(f'  Accuracy  = {opt_row["Accuracy"]:.4f}')

---
## Step 8 · Churn Probability Prediction & Risk Segmentation

**Risk Segmentation Thresholds:**

| Segment | Probability Range | Count | % of Total | Actual Churn Rate |
|---------|------------------|-------|-----------|------------------|
| Low Risk | < 30% | 2,916 | 41.4% | 5.5% |
| Medium Risk | 30%–60% | 912 | 12.9% | 19.4% |
| High Risk | ≥ 60% | 3,215 | 45.6% | **47.7%** |

In [ ]:
from src.prediction import run_prediction
from src.config import CLEANED_DATA_PATH, ID_COL

raw_df = pd.read_csv(CLEANED_DATA_PATH)
cids   = raw_df[ID_COL]
X_all  = pd.concat([X_train, X_test])
y_all  = pd.concat([y_train, y_test])

pred_df = run_prediction(X_all, y_all, cids)

print('Risk Segment Distribution:')
print(pred_df['risk_segment'].value_counts())
print('\nActual churn rate per segment:')
for seg in ['High Risk', 'Medium Risk', 'Low Risk']:
    mask = pred_df['risk_segment'] == seg
    rate = pred_df.loc[mask, 'actual_churn'].mean() * 100
    avg_prob = pred_df.loc[mask, 'churn_probability'].mean() * 100
    print(f'  {seg:12s}: actual churn={rate:.1f}%  avg prob={avg_prob:.1f}%')

display(pred_df.head(8))

In [ ]:
# Probability distribution and risk segment pie
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram
axes[0].hist(pred_df['churn_probability'], bins=50, color='#3b82d4', alpha=0.8, edgecolor='none')
axes[0].axvline(pred_df['churn_probability'].mean(), color='#e05c5c', linestyle='--', lw=2,
                label=f'Mean = {pred_df["churn_probability"].mean():.2f}')
axes[0].set_xlabel('Churn Probability', fontsize=11)
axes[0].set_ylabel('Number of Customers', fontsize=11)
axes[0].set_title('Churn Probability Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Risk segment pie
seg_order  = ['Low Risk', 'Medium Risk', 'High Risk']
seg_colors = ['#3b82d4', '#f59e0b', '#e05c5c']
seg_counts = [len(pred_df[pred_df['risk_segment']==s]) for s in seg_order]
axes[1].pie(seg_counts, labels=seg_order, colors=seg_colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Customer Risk Segmentation', fontsize=12, fontweight='bold')

plt.suptitle('AI Risk Scoring — All 7,043 Customers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 9 · SHAP Explainability (Explainable AI)

SHAP (SHapley Additive exPlanations) provides both **global** and **local** explanations:  
- **Global:** Which features have the largest overall impact on churn probability?
- **Local:** Why did the model score a specific customer as high-risk?

**Top 10 SHAP Drivers (Mean |SHAP|):**

| Rank | Feature | Mean |SHAP| | Direction |
|------|---------|-------------|----------|
| 1 | Contract_Two year | 1.30379 | Protective (reduces churn risk) |
| 2 | InternetService_Fiber optic | 0.74703 | Risk factor |
| 3 | Contract_One year | 0.72822 | Protective |
| 4 | PaymentMethod_Electronic check | 0.33126 | Risk factor |
| 5 | OnlineSecurity_Yes | 0.32818 | Protective |
| 6 | PaperlessBilling | 0.30983 | Risk factor |
| 7 | MultipleLines_Yes | 0.27190 | Risk factor |
| 8 | InternetService_No | 0.26978 | Protective |
| 9 | StreamingMovies_Yes | 0.25870 | Risk factor |
| 10 | StreamingTV_Yes | 0.22897 | Risk factor |

In [ ]:
from src.explainability import run_explainability

# sample_size=300 for notebook speed; full pipeline uses 500-1000
shap_df = run_explainability(X_train, X_test, pred_df, sample_size=300)

print('Top 10 features by mean |SHAP|:')
top_shap = shap_df.abs().mean().sort_values(ascending=False).head(10)
for i, (feat, val) in enumerate(top_shap.items(), 1):
    print(f'  #{i:2d}  {feat:<45s}  {val:.5f}')

In [ ]:
# SHAP global importance bar chart
top15 = shap_df.abs().mean().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 7))
colors_shap = ['#e05c5c' if i < 5 else '#3b82d4' for i in range(len(top15))]
ax.barh(top15.index[::-1], top15.values[::-1], color=colors_shap[::-1], edgecolor='none')
ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax.set_title('SHAP Global Feature Importance — Top 15 Drivers', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Display SHAP beeswarm plot (generated by full pipeline)
ipy_display(Image(os.path.join(PROJECT_ROOT, 'reports', '14_shap_beeswarm.png')))

In [ ]:
# SHAP waterfall for top high-risk customer
ipy_display(Image(os.path.join(PROJECT_ROOT, 'reports', '15_shap_waterfall_customer_1.png')))

---
## Step 10 · Retention Recommendations

A rules-based engine evaluates 11 signal categories per customer and assigns up to 3 prioritised retention actions.  
Customers are ranked by **Priority Score** = `churn_probability × (monthly_charges × remaining_tenure) / 100`

**Top 5 Retention Strategies (by business impact):**
1. Contract upgrade offer (15–20% discount) — targets month-to-month customers
2. Onboarding programme — targets new customers (tenure ≤ 12m)
3. High-value loyalty discount — targets tenure > 24m + charges > $70
4. Auto-pay migration incentive — targets electronic check payers
5. Fiber optic value realignment — targets Fiber optic + charges > $80

In [ ]:
from src.retention_recommendations import run_recommendations

rec_df = run_recommendations(pred_df, raw_df)
print(f'Retention recommendations generated for {len(rec_df):,} at-risk customers')
print()

# Show top 5 by priority score
display(
    rec_df[['customerID','risk_segment','churn_probability',
            'MonthlyCharges','tenure','Recommended Actions','priority_score']]
    .head(5)
    .rename(columns={'Recommended Actions':'recommended_actions'})
)

In [ ]:
# Display retention strategy overview chart
ipy_display(Image(os.path.join(PROJECT_ROOT, 'reports', '17_retention_strategy_overview.png')))

In [ ]:
# Financial impact estimate
high_risk = pred_df[pred_df['risk_segment'] == 'High Risk']
high_risk_with_charges = high_risk.merge(
    raw_df[['customerID','MonthlyCharges']], on='customerID', how='left'
)
avg_charge = high_risk_with_charges['MonthlyCharges'].mean()
revenue_at_risk = avg_charge * 12 * len(high_risk)

print(f'High-risk customers       : {len(high_risk):,}')
print(f'Avg monthly charges       : ${avg_charge:.2f}')
print(f'Annual revenue at risk    : ${revenue_at_risk:,.0f}')
print(f'If 20% are retained       : ${revenue_at_risk * 0.20:,.0f} / year protected')

---
## Step 11 · Power BI Data Export

In [ ]:
from src.powerbi_export import run_powerbi_export

run_powerbi_export(pred_df=pred_df, raw_df=raw_df, rec_df=rec_df)
print('Power BI workbook ready.')

# Confirm sheets
import openpyxl
from src.config import POWERBI_PATH
wb = openpyxl.load_workbook(POWERBI_PATH)
print(f'Sheets in workbook ({len(wb.sheetnames)}):')
for s in wb.sheetnames:
    ws = wb[s]
    print(f'  {s}: {ws.max_row-1:,} rows x {ws.max_column} columns')

---
## Step 12 · Results Summary

### Model Performance (Actual Results)

| Model | ROC-AUC | PR-AUC | F1 | Precision | Recall | Accuracy |
|-------|---------|--------|----|-----------|--------|----------|
| **Logistic Regression** ★ | **0.8461** | **0.6664** | 0.6138 | 0.5079 | 0.7754 | 0.7410 |
| Random Forest | 0.8434 | 0.6552 | 0.6265 | 0.5184 | 0.7914 | 0.7495 |
| XGBoost | 0.8363 | 0.6418 | 0.6198 | 0.5261 | 0.7540 | 0.7544 |

### Key Business Insights

1. **Contract is the strongest churn predictor** — two-year contracts have 2.8% churn vs 42.7% for month-to-month
2. **First-year customers are highest risk** — 47.4% churn in months 0–12 vs only 6.6% after year 5
3. **Fiber optic + electronic check** is a high-risk combination (40%+ churn probability)
4. **3,215 customers (45.6%) are flagged as High Risk** with avg churn probability of 88.3%
5. **Annual revenue at risk: ~$2.5M** — 20% retention saves ~$500K/year

### Output Files Generated

| File | Location | Description |
|------|----------|-------------|
| `predictions.csv` | `outputs/` | 7,043 customers with churn probability and risk segment |
| `shap_values.csv` | `outputs/` | SHAP values for 500 test samples |
| `retention_recommendations.csv` | `outputs/` | 4,127 at-risk customers with actions |
| `powerbi_dashboard_data.xlsx` | `outputs/` | 8-sheet formatted Power BI workbook |
| `optimised_model.pkl` | `models/` | Best fitted model (Logistic Regression) |
| `scaler.pkl` | `models/` | Fitted StandardScaler |
| `optimal_threshold.pkl` | `models/` | Classification threshold (0.57) |
| Charts 01–17 (PNG) | `reports/` | All EDA, model, SHAP, and retention charts |

In [ ]:
# Final verification — check all key output files exist
import os
from src.config import OUTPUTS_DIR, MODELS_DIR, REPORTS_DIR

check_files = [
    os.path.join(OUTPUTS_DIR, 'predictions.csv'),
    os.path.join(OUTPUTS_DIR, 'shap_values.csv'),
    os.path.join(OUTPUTS_DIR, 'retention_recommendations.csv'),
    os.path.join(OUTPUTS_DIR, 'powerbi_dashboard_data.xlsx'),
    os.path.join(MODELS_DIR, 'optimised_model.pkl'),
    os.path.join(MODELS_DIR, 'scaler.pkl'),
    os.path.join(MODELS_DIR, 'optimal_threshold.pkl'),
]

print('Output file verification:')
all_ok = True
for f in check_files:
    exists = os.path.exists(f)
    size_kb = os.path.getsize(f) / 1024 if exists else 0
    status = 'OK' if exists else 'MISSING'
    print(f'  {status}  {os.path.basename(f):<45s} {size_kb:>8.1f} KB')
    if not exists:
        all_ok = False

report_pngs = [f for f in os.listdir(REPORTS_DIR) if f.endswith('.png')]
print(f'\nReport charts: {len(report_pngs)} PNG files in reports/')
print('\nPipeline complete.' if all_ok else '\nWARNING: Some files missing — rerun pipeline steps.')